In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from osgeo import gdal
import os
import pandas as pd
from datetime import datetime, timedelta
import shutil
from glob import glob
from concurrent.futures import ThreadPoolExecutor
import json

In [ ]:
import numpy as np

from sentinelhub import (
    SHConfig,
    CRS,
    BBox,
    DataCollection,
    MimeType,
    MosaickingOrder,
    SentinelHubRequest,
    bbox_to_dimensions,
)

In [ ]:
# Setting Account Info of Sentinel Hub
config = SHConfig()
config.sh_client_id = ""
config.sh_client_secret = ""
config.sh_base_url = "https://sh.dataspace.copernicus.eu"
config.sh_token_url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"

In [ ]:
# Setting request image
# Return number of images
def zero_percentage(array):
    zero_count = np.count_nonzero(array == 0)
    zero_ratio = zero_count / array.size
    return zero_ratio


def download_img(
    betsiboka_coords_wgs84,
    betsiboka_bbox,
    betsiboka_size,
    resolution,
    time_interval,
    dir_name,
):
    cloud_coverage = (0, 10)
    evalscript_all_bands = """
        //VERSION=3
        
        function setup() {
            return {
                input: [{
                    bands: ["B01","B02","B03","B04","B05","B06","B07","B08","B8A","B09","B10","B11","B12"],
                    units: "DN"
                }],
                output: {
                    bands: 13,
                    sampleType: "INT16"
                }
            };
        }
    
        function evaluatePixel(sample) {
            return [sample.B01,
                    sample.B02,
                    sample.B03,
                    sample.B04,
                    sample.B05,
                    sample.B06,
                    sample.B07,
                    sample.B08,
                    sample.B8A,
                    sample.B09,
                    sample.B10,
                    sample.B11,
                    sample.B12];
        }
    """
    request_all_bands = SentinelHubRequest(
        data_folder=dir_name + "/",  # Path for saving images
        evalscript=evalscript_all_bands,
        input_data=[
            SentinelHubRequest.input_data(
                data_collection=DataCollection.SENTINEL2_L1C.define_from(
                    "s2l1c", service_url=config.sh_base_url  # Sentinel-2 L1C Datset
                ),
                time_interval=time_interval,  # time period
                mosaicking_order=MosaickingOrder.LEAST_CC,  # least cloudy acquisitions
                # maxcc=0.1,
            )
        ],
        responses=[
            SentinelHubRequest.output_response("default", MimeType.TIFF)
        ],  # Output image format
        bbox=betsiboka_bbox,  # Interested area
        size=betsiboka_size,  # Image size
        config=config,
    )

    request_all_bands.custom_url_params = {
        "filter": {
            "timeRange": {"from": time_interval[0], "to": time_interval[1]},
            "cloudCoverage": cloud_coverage,
        }
    }

    # save image
    all_bands_response = request_all_bands.get_data(save_data=True)
    # print(all_bands_response)
    # print(type(all_bands_response))
    num = 0
    for single_response in all_bands_response:
        if zero_percentage(single_response[:, :, 0]) < 0.5:
            # single_response.save_data()
            num += 1
    return num

In [ ]:
# One case
x, y = 53.92702, 37.71665
start_time = "2021-01-01T00:00:00"
for i in range(1278):  # 2021-2024.06 1278
    betsiboka_coords_wgs84 = (x - 0.01, y - 0.01, x + 0.01, y + 0.01)
    resolution = 10
    cloud_coverage = (0, 10)  # Cloudy coverage less than 10%
    betsiboka_bbox = BBox(bbox=betsiboka_coords_wgs84, crs=CRS.WGS84)
    betsiboka_size = bbox_to_dimensions(betsiboka_bbox, resolution=resolution)
    print(f"Image shape at {resolution} m resolution: {betsiboka_size} pixels")

    time_obj = datetime.strptime(start_time, "%Y-%m-%dT%H:%M:%S")
    time_obj += timedelta(days=1)
    end_time = time_obj.strftime("%Y-%m-%dT%H:%M:%S")

    time_interval = (start_time, end_time)
    dir_name = "T23/" + start_time[:10]
    # os.mkdir('./'+dir_name)
    num_pic = download_img(
        betsiboka_coords_wgs84,
        betsiboka_bbox,
        betsiboka_size,
        resolution,
        time_interval,
        dir_name,
    )
    if num_pic == 0:
        shutil.rmtree("./ori_data/" + dir_name)
    print(start_time, num_pic)
    start_time = end_time

In [ ]:
# ori_data to dataset
data_csv = pd.DataFrame(columns=["path"])
paths = []
for filepath in os.listdir("./data/raw/"):
    for dates in os.listdir(os.path.join("./raw", filepath)):
        subpath = "./data/raw/" + filepath + "/" + dates
        paths.append(subpath)
data_csv["path"] = paths
data_csv.to_csv("data.csv")

In [ ]:
# Convert *.tiff to *.np
# create dataset/s2
base_path = "./"
for index, row in data_csv.iterrows():
    image_path = glob(base_path + row["path"] + "/**/*.tiff")[0]
    img_data = gdal.Open(image_path)
    cols = img_data.RasterXSize
    rows = img_data.RasterYSize

    s2_data = np.zeros((13, rows, cols), dtype=np.float32)
    for i in range(13):
        band = img_data.GetRasterBand(i + 1)  # 波段索引从1开始
        s2_data[i, :, :] = band.ReadAsArray()
    np.save("./data/processed/s2/" + str(index) + ".npy", s2_data)
    print(row["path"])

In [ ]:
# filter cloudy
# create dataset/RGB
def nor_img(channel_data):
    min_value = channel_data.min()
    max_value = channel_data.max()
    normalized_data = (channel_data - min_value) / (max_value - min_value)
    return normalized_data


for i in range(16918):
    res = np.load("./data/processed/s2/" + str(i) + ".npy")
    channel_data2 = nor_img(res[4])
    channel_data3 = nor_img(res[3])
    channel_data4 = nor_img(res[2])
    channel_data = np.dstack((channel_data2, channel_data3, channel_data4))
    plt.imshow(channel_data)
    plt.title(i)
    plt.savefig("./data/processed/RGB/" + str(i) + ".jpg")

data_csv_filled = data_csv.fillna(0)
data_csv_filled.head()
data_csv_filled.to_csv("data_label.csv", index=False)

In [ ]:
# Generate Initial plume (MBMP)
# Create dataset/detection/*.npy
def band_ratio_w_hm(data):
    band12 = data.GetRasterBand(13).ReadAsArray()
    band11 = data.GetRasterBand(12).ReadAsArray()
    c = band11.mean() / band12.mean()
    if band11.min() == 0:
        band11 += 1
    if band12.min() == 0:
        band12 += 1
    return np.log(c * band12 / band11)


data_cloud_free = data_csv_filled[data_csv_filled["cloudy"] == 0]
base_path = "./"
for i in range(len(data_cloud_free) - 1):
    current_row = data_cloud_free.iloc[i]
    next_row = data_cloud_free.iloc[i + 1]
    if (
        current_row["path"].split("/")[-1].split("-")[0] == "2024"
        and next_row["path"].split("/")[-1].split("-")[0] == "2021"
    ):
        continue
    index = current_row["index"]
    data = gdal.Open(glob(base_path + current_row["path"] + "/**/*.tiff")[0])
    data1 = gdal.Open(glob(base_path + next_row["path"] + "/**/*.tiff")[0])

    res = band_ratio_w_hm(data1) - band_ratio_w_hm(data)
    np.save("./data/processed/detection/" + str(index) + ".npy", res)
    print(current_row["path"])

In [ ]:
# Create dataset/detection_RGB/*.jpg for plume confirmation
base_path = "./data/processed/detection/"
for filepath in os.listdir(base_path):
    res = np.load(os.path.join(base_path, filepath))
    filename = filepath.split(".")[0]
    plt.imsave(
        f"./data/processed/detection_RGB/{filename}.jpg",
        res,
        cmap="RdBu_r",
        vmin=-0.05,
        vmax=0.05,
    )

In [ ]:
# Deal with batch preprocessing
def process_image(i):
    try:
        res = np.load(f"./data/processed/s2/{i}.npy")
        channel_data2 = nor_img(res[4])
        channel_data3 = nor_img(res[3])
        channel_data4 = nor_img(res[2])
        channel_data = np.dstack((channel_data2, channel_data3, channel_data4))
        plt.imsave(f"./data/processed/RGB/{i}.jpg", channel_data)
        print(f"Image {i} processed and saved.")
    except Exception as e:
        print(f"Failed to process image {i}: {e}")


start = 0
end = 16918

with ThreadPoolExecutor(max_workers=64) as executor:
    futures = [executor.submit(process_image, i) for i in range(start, end)]
    for future in futures:
        future.result()

print("All images processed.")

In [ ]:
# For labeled plume generation
target_dir = "../../data"
source_dir = "./"

data_label = pd.read_csv("./data_label.csv")
data_labeled = data_label[data_label["plume"] != 0]

for index, row in data_labeled.iterrows():
    target_file_path = "./data/processed/label/" + str(row["index"]) + ".jpg"
    source_file_path = "./data/processed/detection_RGB/" + str(row["index"]) + ".jpg"
    shutil.copy(source_file_path, target_file_path)

In [ ]:
# Extract and Generate mask
with open("toras_anns_add.json", "r") as f:
    data = json.load(f)
data_details = pd.read_csv("./data_details.csv")

# Positive samples
for i in range(len(data)):
    image_name = data[i]["documents"][0]["name"]
    print(image_name)
    annotations = data[i]["annotation"]["annotationGroups"][0]["annotationEntities"][0][
        "annotationBlocks"
    ][0]["annotations"]
    image_index = int(image_name.split(".")[0])

    height, width = (
        data_details.iloc[image_index]["width"],
        data_details.iloc[image_index]["height"],
    )
    mask = np.zeros((height, width), dtype=np.uint8)
    fig, ax = plt.subplots()
    ax.imshow(mask, cmap="gray")

    for segment in annotations:
        coordinates = segment["segments"][0]
        polygon = Polygon(
            coordinates, closed=True, fill=True, edgecolor="white", facecolor="white"
        )
        ax.add_patch(polygon)

    ax.axis("off")
    plt.savefig(
        "./data/processed/mask/" + image_name,
        bbox_inches="tight",
        pad_inches=0,
        dpi=100,
    )

# Negative samples
data_labels = pd.read_csv("./data_label.csv")
non_cloudy_plume = data_labels[data_labels["cloudy"] == 0][data_labels["plume"] != 2]
for index, row in non_cloudy_plume.iterrows():
    height, width = row["width"], row["height"]
    mask = np.zeros((height, width), dtype=np.uint8)
    image_name = str(row["index"]) + ".jpg"
    plt.imsave(
        f"./data/processed/mask/{image_name}",
        mask,
        cmap="gray",
        format="png",
        vmin=0,
        vmax=255,
    )

In [ ]:
# Split dataset
data_label = pd.read_csv("./data_label.csv")
data_label.head()
data_label["year"] = data_label["path"].apply(lambda x: x.split("/")[-1].split("-")[0])

test = data_label[data_label["cloudy"] == 0][data_label["year"] == "2024"][
    ["index", "plume", "width", "height"]
]
test = test.drop(test.index[-1])
test.to_csv("./test.csv", index=False)

In [ ]:
train_val_rows = data_label[data_label["cloudy"] == 0][data_label["year"] != "2024"]
train_rows = train_val_rows.sample(frac=0.8, random_state=42)
train = train_rows[["index", "plume", "width", "height"]]
val_rows = train_val_rows.drop(train_rows.index)
val = val_rows[["index", "plume", "width", "height"]]
train.to_csv("./train.csv", index=False)
val.to_csv("./val.csv", index=False)

In [ ]:
# Build training, validation, test dataset
list2 = list(data_label[data_label["cloudy"] == 0]["index"])
train_pos = [list2.index(x) for x in train["index"]]
new_positions = [pos - 1 for pos in train_pos]
train_pre_index = [list2[pos] for pos in new_positions]
train["pre_index"] = train_pre_index
new_positions = [pos + 1 for pos in train_pos]
train_next_index = [list2[pos] for pos in new_positions]
train["next_index"] = train_next_index
train.to_csv("./train.csv", index=False)